In [1]:
import re
import shutil
from pathlib import Path

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# 1. Find project root by looking for pyproject.toml (works from any cwd)
def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise FileNotFoundError("Couldn't find pyproject.toml above " + str(start))

ROOT_DIR = find_root(Path.cwd())
DATA_DIR = ROOT_DIR / "data"
DB_DIR = ROOT_DIR / "src"  / "chroma_store"
REBUILD = True   # set to False once it works

print("Data dir:", DATA_DIR, "| exists:", DATA_DIR.exists())
print("Files:", [f.name for f in DATA_DIR.iterdir()][:3])

# 2. Load transcripts (accepts .vtt / .srt / .txt, any case)
def load_transcripts():
    files = [f for f in DATA_DIR.iterdir() if f.suffix.lower() in {".vtt", ".srt", ".txt"}]
    if not files:
        raise FileNotFoundError(f"No transcript files in {DATA_DIR}")

    docs = []
    for path in files:
        lines = []
        for line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
            line = line.strip()
            if (not line or line == "WEBVTT" or "-->" in line
                    or line.isdigit() or line.startswith(("NOTE", "Kind:", "Language:"))):
                continue
            line = re.sub(r"<[^>]+>", "", line)       # strip <c> tags
            if lines and lines[-1] == line:            # drop repeated caption lines
                continue
            lines.append(line)

        match = re.search(r"Session\s*(\d+)", path.name, re.IGNORECASE)
        session = match.group(1) if match else "unknown"
        docs.append(Document(page_content=" ".join(lines),
                             metadata={"session": session, "source": path.name}))
    return docs

# 3. Build / load store
def load_store():
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    if REBUILD and DB_DIR.exists():
        shutil.rmtree(DB_DIR)

    if DB_DIR.exists() and any(DB_DIR.iterdir()):
        store = Chroma(persist_directory=str(DB_DIR), embedding_function=embeddings)
        if store._collection.count() > 0:
            print("Loaded existing store:", store._collection.count(), "chunks")
            return store
        print("Existing store is empty, rebuilding...")
        del store
        shutil.rmtree(DB_DIR)

    docs = load_transcripts()
    print(f"Loaded {len(docs)} documents")
    chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150).split_documents(docs)
    print(f"Split into {len(chunks)} chunks")
    return Chroma.from_documents(chunks, embeddings, persist_directory=str(DB_DIR))

store = load_store()
retriever = store.as_retriever(search_kwargs={"k": 5})

query = "what is regression testing?"
for r in retriever.invoke(query):
    print(f"[Session {r.metadata['session']}] {r.page_content[:150]}...\n")

Data dir: c:\Users\gurpr\OneDrive\Desktop\RAG-EVAL\data | exists: True
Files: ['YT Sandbox   LLM Evals Session 1.vtt', 'YT Sandbox   LLM Evals Session 2.vtt', 'YT Sandbox   LLM Evals Session 3.vtt']


c:\Users\gurpr\OneDrive\Desktop\RAG-EVAL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6499.28it/s]


Loaded 8 documents
Split into 693 chunks
[Session 8] now what we are doing we are building our eval suite we are writing different evaluation codes for different different components and pipelines and ap...

[Session 8] file is telling that your new retriever is better than the previous one the new generator is telling that the new generator is better than the previou...

[Session 3] running it on multiple versions of the software. This is the second benefit. Third is regression testing. This is also a very important benefit. Does ...

[Session 8] so that is why its adoption is also more and there is a good chance that after a year or two this NLM evaluations will become a benchmark library it w...

[Session 4] the exam and during the exam you fix some things like which prompt you will send, what kind of conditions will be there you fix all these things so th...



In [ ]:
# RETRIEVER LOGIC:
# This notebook builds a retrieval pipeline over the YouTube session transcripts in /data.
# First, load_transcripts() reads each .vtt file, strips out the WEBVTT header, timestamps,
# and duplicate caption lines, and joins the remaining text into one Document per session.
# The session number is pulled from the filename with a regex and saved in the Document's
# metadata along with the filename, which acts as the citation. Next, load_store() splits
# each Document into overlapping 1000-character chunks (150-character overlap, so ideas that
# straddle a boundary aren't lost). Each chunk keeps its parent's metadata. The chunks are
# embedded with the all-MiniLM-L6-v2 model and persisted in a Chroma vector database
# (chroma_store), so later runs can load the existing store instead of re-embedding.
# Finally, store.as_retriever(k=5) wraps the database in a retriever: when given a query,
# it embeds the query with the same model, finds the 5 chunks whose embeddings are closest
# in meaning, and returns them as Document objects. These are stored in a list that can be
# looped over to read each chunk's text and its session/source metadata for citations.